# AI文章検出ツール ― Binoculars vs 統計的手法 比較ノートブックこのノートブックでは、学生のレポートなどのテキストがAIで生成されたものかを **段落ごと** に判定します。| 手法 | 概要 ||---|---|| **統計的手法** | 文長均一性・語彙多様性・接続詞頻度・句読点パターンの4指標 || **Binoculars** | 2つのLLM（Observer / Performer）のパープレキシティ比率 |**実行環境**: Google Colab（T4 GPU 推奨、CPUでもフォールバック動作）

## 1. セットアップ

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes sentencepiece protobufimport warningswarnings.filterwarnings("ignore")print("セットアップ完了")

## 2. テキスト入力分析したいテキストを `sample_text` に貼り付けてください。段落は空行で区切ります。

In [ ]:
import re# ── ここにテキストを貼り付け ──sample_text = """人工知能の発展は、現代社会に大きな影響を与えている。特に、自然言語処理の分野では、大規模言語モデルの登場により、テキスト生成の品質が飛躍的に向上した。これにより、教育現場においても、AIが生成した文章と人間が書いた文章を区別することが重要な課題となっている。一方で、従来の文章作成においては、個人の経験や感情が反映されることが多く、文体にも独自の特徴が現れる。例えば、句読点の打ち方や接続詞の使い方には、書き手の癖が出やすい。こうした特徴は、AIが生成する文章には見られにくい傾向がある。昨日友達とカフェに行って、めっちゃ美味しいケーキ食べた。チョコレートのやつが最高だった！また行きたいな〜。さらに、AI生成テキストの特徴として、文の長さが均一であること、語彙の選択が平均的であること、そして論理展開が整然としていることが挙げられる。したがって、これらの統計的特徴を分析することで、AI生成テキストを検出できる可能性がある。加えて、近年の研究では、言語モデル自体を用いた検出手法も提案されている。つまり、教育現場では複数の検出手法を組み合わせることが望ましい。具体的には、統計的分析とAIモデルベースの分析を併用することで、より信頼性の高い判定が可能になると考えられる。"""# ── 段落分割 ──MIN_PARAGRAPH_LENGTH = 50def split_paragraphs(text):    normalized = text.replace("\r\n", "\n").strip()    if not normalized:        return []    chunks = re.split(r"\n\s*\n+", normalized)    if len(chunks) == 1:        chunks = normalized.split("\n")    result = []    for chunk in chunks:        t = chunk.strip()        if not t:            continue        result.append({"index": len(result), "text": t, "skipped": len(t) < MIN_PARAGRAPH_LENGTH})    return resultparagraphs = split_paragraphs(sample_text)for p in paragraphs:    status = "SKIP" if p["skipped"] else "OK"    print(f'[段落{p["index"]+1}] {status} ({len(p["text"])}文字): {p["text"][:40]}...')print(f"\n合計: {len(paragraphs)}段落 (分析対象: {sum(1 for p in paragraphs if not p['skipped'])})")

## 3. 統計的手法（Webアプリ移植版）Webアプリ (`lib/statisticalAnalysis.ts`) と同一のアルゴリズムをPythonに移植。| 指標 | AI的なテキストの特徴 ||---|---|| 文長均一性 | 文の長さのばらつきが小さい || 語彙多様性 (TTR) | TTRが中程度（0.4〜0.6付近） || 接続詞頻度 | 接続詞を多用する || 句読点規則性 | 句読点の間隔が等間隔に近い |

In [ ]:
import numpy as npCONJUNCTIONS = [    "また", "さらに", "一方で", "しかし", "そのため",    "したがって", "つまり", "すなわち", "加えて", "それに加えて",    "具体的には", "例えば", "このように", "以上のことから", "結果として",    "特に", "なお", "ただし", "もっとも", "それゆえ",]def split_sentences(text):    return [s.strip() for s in re.split(r"[。！？\n]+", text) if s.strip()]def tokenize(text):    tokens = re.findall(r"[a-zA-Z0-9]+", text)    japanese = re.sub(r"[a-zA-Z0-9\s\u0000-\u007F]", "", text)    for i in range(len(japanese) - 1):        tokens.append(japanese[i:i+2])    return tokensdef analyze_sentence_length_uniformity(sentences):    if len(sentences) < 2:        return 0.5    lengths = [len(s) for s in sentences]    mean = np.mean(lengths)    if mean == 0:        return 0.5    cv = np.std(lengths) / mean    return float(max(0, min(1, 1 - cv / 0.8)))def analyze_vocabulary_diversity(text):    tokens = tokenize(text)    if len(tokens) < 10:        return 0.5    ttr = len(set(tokens)) / len(tokens)    distance = abs(ttr - 0.5)    return float(max(0, min(1, 1 - distance / 0.35)))def analyze_conjunction_frequency(text, sentences):    if not sentences:        return 0.5    count = sum(len(re.findall(c, text)) for c in CONJUNCTIONS)    rate = count / len(sentences)    return float(max(0, min(1, rate / 0.5)))def analyze_punctuation_regularity(text):    positions = [i for i, ch in enumerate(text) if ch in "、，"]    if len(positions) < 3:        return 0.5    gaps = np.diff(positions)    mean = np.mean(gaps)    if mean == 0:        return 0.5    cv = np.std(gaps) / mean    return float(max(0, min(1, 1 - cv / 0.8)))def analyze_statistically(text):    sentences = split_sentences(text)    slu = analyze_sentence_length_uniformity(sentences)    vd = analyze_vocabulary_diversity(text)    cf = analyze_conjunction_frequency(text, sentences)    pr = analyze_punctuation_regularity(text)    overall = (slu + vd + cf + pr) / 4    return {"slu": slu, "vd": vd, "cf": cf, "pr": pr, "overall": overall}# テストfor p in paragraphs:    if p["skipped"]:        continue    r = analyze_statistically(p["text"])    print(f'段落{p["index"]+1}: 統計スコア={r["overall"]:.3f}')    print(f'  文長均一性={r["slu"]:.3f}, 語彙多様性={r["vd"]:.3f}, 接続詞={r["cf"]:.3f}, 句読点={r["pr"]:.3f}')    print()

## 4. Binoculars（LLMベース検出）**論文**: "Spotting LLMs With Binoculars: Zero-Shot Detection of Machine-Generated Text"**原理**:- Observer（汎用LLM）と Performer（instruction-tuned LLM）の2つのモデルを使用- 同じテキストに対する両モデルのクロスエントロピーを比較- AI生成テキスト → 両モデルで似た予測 → 比率が1に近い → AI的- 人間のテキスト → Observerの予測が悪い → 比率が大きい → 人間的

In [ ]:
import torchdef get_device():    if torch.cuda.is_available():        name = torch.cuda.get_device_name(0)        vram = torch.cuda.get_device_properties(0).total_mem / 1e9        print(f"GPU検出: {name} (VRAM: {vram:.1f} GB)")        return "cuda", vram    print("GPU未検出 → CPUモード（軽量モデルを使用）")    return "cpu", 0DEVICE, VRAM = get_device()USE_LARGE = DEVICE == "cuda" and VRAM > 10if USE_LARGE:    OBSERVER_NAME = "tiiuae/falcon-7b"    PERFORMER_NAME = "tiiuae/falcon-7b-instruct"    print(f"大規模モデル: {OBSERVER_NAME} / {PERFORMER_NAME} (4bit量子化)")else:    OBSERVER_NAME = "openai-community/gpt2"    PERFORMER_NAME = "openai-community/gpt2-medium"    print(f"軽量モデル: {OBSERVER_NAME} / {PERFORMER_NAME}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfigdef load_model(name):    print(f"ロード中: {name} ...", end=" ", flush=True)    kwargs = {"device_map": "auto"} if DEVICE == "cuda" else {}    if USE_LARGE:        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")        model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, trust_remote_code=True, **kwargs)    else:        model = AutoModelForCausalLM.from_pretrained(name, **kwargs)        if DEVICE == "cuda":            model = model.half()    model.eval()    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)    if tok.pad_token is None:        tok.pad_token = tok.eos_token    print("完了")    return model, tokobserver_model, observer_tok = load_model(OBSERVER_NAME)performer_model, performer_tok = load_model(PERFORMER_NAME)

In [ ]:
def compute_ce(model, tokenizer, text, max_length=512):    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)    ids = inputs["input_ids"].to(model.device)    if ids.shape[1] < 2:        return float("nan")    with torch.no_grad():        out = model(ids, labels=ids)    return out.loss.item()def binoculars_score(text):    obs = compute_ce(observer_model, observer_tok, text)    perf = compute_ce(performer_model, performer_tok, text)    if np.isnan(obs) or np.isnan(perf) or perf == 0:        return 0.5    return obs / perfdef normalize_bino(raw):    # raw ≈ 1.0 → AI, raw > 1.3 → 人間    LOW, HIGH = 0.9, 1.4    prob = 1.0 - (raw - LOW) / (HIGH - LOW)    return float(max(0.0, min(1.0, prob)))print("Binocularsスコア計算中...\n")for p in paragraphs:    if p["skipped"]:        continue    raw = binoculars_score(p["text"])    norm = normalize_bino(raw)    print(f'段落{p["index"]+1}: 生スコア={raw:.4f}, AI確率={norm:.3f}')

## 5. 比較分析両手法のスコアを並べて比較します。- **統計的手法**: 文体の特徴量ベース（軽量・高速）- **Binoculars**: LLMの内部表現ベース（高精度・GPU必要）- **総合スコア**: Binoculars 70% + 統計 30% のハイブリッド

In [ ]:
import pandas as pddef classify(prob):    if prob >= 0.75:        return "AI"    elif prob >= 0.40:        return "混在"    return "人間"rows = []for p in paragraphs:    if p["skipped"]:        rows.append({"段落": p["index"]+1, "テキスト": p["text"][:30]+"...",                      "統計": None, "Binoculars": None, "総合": None, "判定": "スキップ"})        continue    stat = analyze_statistically(p["text"])    braw = binoculars_score(p["text"])    bprob = normalize_bino(braw)    comb = bprob * 0.7 + stat["overall"] * 0.3    rows.append({"段落": p["index"]+1, "テキスト": p["text"][:30]+"...",                  "統計": round(stat["overall"], 3), "Binoculars": round(bprob, 3),                  "総合": round(comb, 3), "判定": classify(comb)})df = pd.DataFrame(rows)display(df)

## 6. 可視化

In [ ]:
import matplotlib.pyplot as pltimport matplotlibmatplotlib.rcParams["font.family"] = "DejaVu Sans"analyzed = df[df["判定"] != "スキップ"].copy()x = analyzed["段落"].valuesfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Bar chartax = axes[0]w = 0.25ax.bar(x - w, analyzed["統計"], w, label="Statistical", color="#60a5fa")ax.bar(x, analyzed["Binoculars"], w, label="Binoculars", color="#f97316")ax.bar(x + w, analyzed["総合"], w, label="Combined", color="#a855f7")ax.axhline(0.75, color="red", ls="--", alpha=0.5, label="AI (0.75)")ax.axhline(0.40, color="#eab308", ls="--", alpha=0.5, label="Mixed (0.40)")ax.set_xlabel("Paragraph")ax.set_ylabel("AI Score")ax.set_title("Method Comparison")ax.set_xticks(x)ax.legend(fontsize=8)ax.set_ylim(0, 1.05)# Scatterax = axes[1]sc = ax.scatter(analyzed["統計"], analyzed["Binoculars"], s=100,                c=analyzed["総合"], cmap="RdYlGn_r", edgecolors="black", lw=0.5, vmin=0, vmax=1)for _, r in analyzed.iterrows():    ax.annotate(f'P{int(r["段落"])}', (r["統計"], r["Binoculars"]), xytext=(5,5), textcoords="offset points", fontsize=9)ax.plot([0,1],[0,1],"k--",alpha=0.3)ax.set_xlabel("Statistical")ax.set_ylabel("Binoculars")ax.set_title("Correlation")ax.set_xlim(0,1)ax.set_ylim(0,1)plt.colorbar(sc, ax=ax, label="Combined")plt.tight_layout()plt.show()

## 7. 色付き段落表示 + 総合判定

In [ ]:
from IPython.display import HTMLdef s2c(s):    if s is None: return "#f3f4f6", "#9ca3af"    if s >= 0.75: return "#fecaca", "#ef4444"    if s >= 0.40: return "#fef08a", "#eab308"    return "#bbf7d0", "#22c55e"def s2l(s):    if s is None: return "スキップ"    if s >= 0.75: return "AIの可能性が高い"    if s >= 0.40: return "判定が混在"    return "人間が書いた可能性が高い"def fp(v):    return "-" if v is None else f"{v:.0%}"parts = ['<div style="font-family:sans-serif;max-width:800px">']parts.append("<h2>段落ごとの判定結果</h2>")for _, row in df.iterrows():    c = row["総合"]    bg, bd = s2c(c)    idx = int(row["段落"])    ptxt = [p for p in paragraphs if p["index"]+1==idx][0]["text"]    parts.append(        '<div style="background:'+bg+';border-left:4px solid '+bd        +';padding:12px 16px;margin:8px 0;border-radius:8px">'        '<div style="display:flex;justify-content:space-between;margin-bottom:6px">'        '<span style="font-size:12px;color:#64748b"><b>段落 '+str(idx)+'</b> — '+s2l(c)+'</span>'        '<span style="font-size:11px;color:#64748b">統計:'+fp(row["統計"])        +' | Bino:'+fp(row["Binoculars"])+' | <b>総合:'+fp(c)+'</b></span></div>'        '<p style="font-size:14px;color:#334155;line-height:1.7;margin:0;white-space:pre-wrap">'        +ptxt+'</p></div>'    )v = df[df["判定"]!="スキップ"]ac = v["総合"].mean()abg, abd = s2c(ac)parts.append(    '<div style="background:'+abg+';border:2px solid #94a3b8;padding:16px;margin-top:20px;border-radius:12px">'    '<h3 style="margin:0 0 8px">全体の総合判定: '+s2l(ac)+' ('+fp(ac)+')</h3>'    '<table style="font-size:14px">'    '<tr><td style="padding-right:20px">統計的手法（平均）</td><td><b>'+fp(v["統計"].mean())+'</b></td></tr>'    '<tr><td>Binoculars（平均）</td><td><b>'+fp(v["Binoculars"].mean())+'</b></td></tr>'    '<tr><td>総合スコア（平均）</td><td><b>'+fp(ac)+'</b></td></tr>'    '<tr><td>AI判定</td><td><b>'+str(len(v[v["判定"]=="AI"]))+'</b> / '+str(len(v))+'</td></tr>'    '<tr><td>混在判定</td><td><b>'+str(len(v[v["判定"]=="混在"]))+'</b> / '+str(len(v))+'</td></tr>'    '<tr><td>人間判定</td><td><b>'+str(len(v[v["判定"]=="人間"]))+'</b> / '+str(len(v))+'</td></tr>'    '</table>'    '<p style="font-size:11px;color:#64748b;margin-top:8px">判定方式: Binoculars (70%) + 統計的手法 (30%)</p>'    '</div>')parts.append("</div>")display(HTML("\n".join(parts)))